In [1]:
import holoviews as hv
import panel as pn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
pn.extension()
hv.extension('bokeh')

The geometry of the problem is illustrated below. Note that the angles AOP and BOP are always 90 degrees (i.e. the lines AP and BP are always tangents to the circle). The point P is distance h away from the edge of the circle, so the red dashed line has length R+h.<br>
The partial circumference of the circle from A to C (through B) plus the segments AP and CP comprise the circumference of the circle, plus the extra rope (x), or 2πR + x

<img src="../img/geometry.png" width="50%">

From geometry:<br>
cos(θ) = R/(R+h) and thus θ = arccos(R/(R+h))<br>
Also: R*sec(θ) = (R+h) (sec = 1/cos)

The length of AP (and CP) can be found by Pythagoras: sqrt([R+h]**2 - R**2)<br>
Let's call t = sqrt([R+h]^2 - R^2)<br>

The length d of the rope is: AP + CP - the arc-length between A and C:<br>
   d = 2t - 2Rarccos(R/(R+0h))<br>
substituting gives:
   d = 2*(sqrt([R+h]^2 - R^2)) - 2Rarccos(R/(R+0h))<br>
with (R+h) = R*sec(θ):<br>
   d = 2*sqrt([Rsec(θ)]^2 - R^2) - 2Rθ<br>
   d = 2*sqrt(R^2sec^2(θ) - R^2) - 2Rθ<br>
   d = 2R*sqrt(sec^2(θ) - 1) - 2Rθ<br>
as:
   sqrt(sec^2(θ) - 1) = tan(θ)
then:
   d = 2R*tan(θ) - 2Rθ<br>
   d/2R = tan(θ) - θ<br>
<br>
When d is given, there is only one solution, and 0 <= θ <= 90 degrees (or 0 <= θ <= π//2)

In [ ]:
# ------------------------------------------------------------
# Calculate h from R and the extra rope length
# ------------------------------------------------------------
def calculate_h(R, extra_length):
    if extra_length <= 0:
        return 0.0
    def rope_equation(h):
        theta = np.arccos(R / (R + h))
        tangent = np.sqrt((R + h)**2 - R**2)
        # Extra rope = two tangent sections minus the
        # piece of circumference that they replace
        return 2 * (tangent - R * theta) - extra_length
    # Find h
    return brentq(rope_equation, 0, extra_length + 10 * np.sqrt(R * extra_length + 1))


# ------------------------------------------------------------
# Plot geometry
# ------------------------------------------------------------
def rope_geometry(R, extra_length):
    h = calculate_h(R, extra_length)
    # Angle between OP and OA/OB
    theta = np.arccos(R / (R + h))
    # Tangent points
    x_t = R * np.sin(theta)
    y_t = R * np.cos(theta)
    # Point where rope is pulled
    P = (0, R + h)

    # --------------------------------------------------------
    # Circle (dashed, full)
    # --------------------------------------------------------
    phi = np.linspace(0, 2 * np.pi, 500)
    x_circle = R * np.cos(phi)
    y_circle = R * np.sin(phi)
    circle = hv.Curve((x_circle, y_circle), label="circle").opts(line_dash="dashed", line_width=1, color="gray")

    # --------------------------------------------------------
    # Rope along the circle
    # Keep the MAJOR arc (around the bottom)
    # --------------------------------------------------------
    phi_rope = np.linspace(theta, 2 * np.pi - theta, 500)
    y_rope_arc = R * np.cos(phi_rope)
    x_rope_arc = R * np.sin(phi_rope)
    rope_arc = hv.Curve((x_rope_arc, y_rope_arc), label="rope around circle").opts(line_width=3, color="brown")

    # --------------------------------------------------------
    # Tangent sections
    # --------------------------------------------------------
    x_left = np.linspace(-x_t, 0, 100)
    x_right = np.linspace(0, x_t, 100)
    # Straight lines from tangent points to P
    y_left = y_t + (R + h - y_t) * (x_left + x_t) / x_t
    y_right = y_t + (R + h - y_t) * (x_t - x_right) / x_t
    tangent_left = hv.Curve((x_left, y_left)).opts(line_width=3, color="brown")
    tangent_right = hv.Curve((x_right, y_right)).opts(line_width=3, color="brown")

    # --------------------------------------------------------
    # Radius lines OA, OB and OP (dotted construction lines)
    # --------------------------------------------------------
    oa_line = hv.Curve(([0, -x_t], [0, y_t])).opts(line_dash="dotted", line_width=1, color="gray")
    ob_line = hv.Curve(([0, x_t], [0, y_t])).opts(line_dash="dotted", line_width=1, color="gray")
    op_line = hv.Curve(([0, 0], [0, R + h])).opts(line_dash="dotted", line_width=1, color="gray")

    # --------------------------------------------------------
    # Radius R (solid, drawn on top of OB)
    # --------------------------------------------------------
    radius_line = hv.Curve(([0, x_t], [0, y_t])).opts(line_width=2, color="black")

    # --------------------------------------------------------
    # Points
    # --------------------------------------------------------
    points = hv.Scatter([(0, 0), (0, R + h), (-x_t, y_t), (x_t, y_t)]).opts(size=8, color="black")

    # --------------------------------------------------------
    # Labels
    # --------------------------------------------------------
    labels = hv.Labels(
        [(0.05 * R, -0.08 * R, "O"),
         (0.05 * R, R + h, "P"),
         (-x_t - 0.12 * R, y_t, "B"),
         (x_t + 0.05 * R, y_t, "A"),
         (x_t / 2, y_t / 2, f"R = {R:.2f} m")],
        kdims=["x", "y"], vdims=["text"]).opts(text_font_size="11pt")

    # --------------------------------------------------------
    # Height h annotation (arrow + label)
    # --------------------------------------------------------
    h_arrow = hv.Segments([(0.75 * R, R, 0.75 * R, R + h)]).opts(color="black", line_width=1)
    h_label = hv.Labels([(0.8 * R, R + h / 2, f"h = {h:.3f} m")], kdims=["x", "y"], vdims=["text"]).opts(text_font_size="12pt", text_align="left")

    # --------------------------------------------------------
    # Combine
    # --------------------------------------------------------
    margin = max(0.2 * R, 0.2 * h, 0.2)
    overlay = (
        circle
        * rope_arc
        * tangent_left
        * tangent_right
        * oa_line
        * ob_line
        * op_line
        * radius_line
        * points
        * labels
        * h_arrow
        * h_label
    )

    title = (
        f"Rope pulled from a circle   "
        f"Extra rope = {extra_length:.3f} m,   h = {h:.3f} m"
    )

    # --------------------------------------------------------
    # Size the frame in proportion to the data extents so that
    # 1 unit in x equals 1 unit in y (a true 1:1 aspect).
    #
    # Using aspect="equal" together with a *fixed square* frame
    # forces Bokeh to squeeze/expand one axis to fit, and it does
    # so anchored at one edge rather than symmetrically -- which
    # is what made the circle drift left whenever h > 0. Sizing
    # the frame itself to match the data's true width:height
    # ratio avoids that correction entirely.
    # --------------------------------------------------------

    data_w = (R + margin) - (-R - margin)
    data_h = (R + h + margin) - (-R - margin)

    max_px = 600
    frame_width = max_px if data_w >= data_h else round(max_px * data_w / data_h)
    frame_height = max_px if data_h >= data_w else round(max_px * data_h / data_w)

    return overlay.opts(
        hv.opts.Overlay(
            title=title,
            frame_width=frame_width,
            frame_height=frame_height,
            xlabel="x [m]",
            ylabel="y [m]",
            xlim=(-R - margin, R + margin),
            ylim=(-R - margin, R + h + margin),
            show_grid=True,
            gridstyle={"grid_line_alpha": 0.2},
            legend_position="top_left",
        )
    )

# ------------------------------------------------------------
# Widgets
# ------------------------------------------------------------
extra_length = pn.widgets.FloatInput(name="Extra rope length [m]", value=1.0, step=0.1, start=0.0, end=10)
radius = pn.widgets.FloatSlider(name="Circle radius R [m]", value=1.0, step=0.1, start=0.1, end=10)

# ------------------------------------------------------------
# Bind plot to widgets
# ------------------------------------------------------------
plot = pn.bind(rope_geometry, radius, extra_length)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------
pn.Column(pn.Row(radius, extra_length), plot).show()

Launching server at http://localhost:61920


C:\Users\HARBR\AppData\Local\Temp\ipykernel_30928\430795040.py:53: RuntimeWarning: invalid value encountered in divide
  y_left = y_t + (R + h - y_t) * (x_left + x_t) / x_t
C:\Users\HARBR\AppData\Local\Temp\ipykernel_30928\430795040.py:54: RuntimeWarning: invalid value encountered in divide
  y_right = y_t + (R + h - y_t) * (x_t - x_right) / x_t


: 